In [ ]:
#Alameda, CA (06001C_20241119)
import os
import zipfile
import glob
import geopandas as gpd
from shapely.geometry import Point
import folium
from folium.features import GeoJsonTooltip, GeoJsonPopup

# -------------------------------------------------------------------
# 1. EXTRACT FEMA ZIP (IF NEEDED)
# -------------------------------------------------------------------
zip_path = './FEMA_Downloads/06115C_20240605.zip'  # Update if needed
extract_path = './FEMA_Downloads/06001C_20241119'

# Check if the zip file exists
if not os.path.exists(zip_path):
    raise FileNotFoundError(f"Zip file not found: {zip_path}")

if not os.path.exists(extract_path):
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(path=extract_path)
    print(f"Extracted files to {extract_path}")
else:
    print(f"Using existing extracted folder: {extract_path}")

# -------------------------------------------------------------------
# 2. CREATE A GEODATAFRAME FOR LAT/LON POINTS
# -------------------------------------------------------------------
lat_lon_points = [
    (34.052235, -118.243683),  # Los Angeles
    (37.774929, -122.419418),  # San Francisco
]

points_gdf = gpd.GeoDataFrame(
    geometry=[Point(lon, lat) for lat, lon in lat_lon_points],
    crs="EPSG:4326"
)

# -------------------------------------------------------------------
# 3. LOCATE SHAPEFILES
# -------------------------------------------------------------------
shapefile_paths = glob.glob(os.path.join(extract_path, "*.shp"))

if not shapefile_paths:
    print("No shapefiles found in the extraction folder.")
else:
    print(f"Found {len(shapefile_paths)} shapefile(s) in {extract_path}")

# -------------------------------------------------------------------
# 4. FLOOD ZONE EXPLANATIONS & COLORS
# -------------------------------------------------------------------
future_color = "#ffa500"
future_label = "Future Conditions 1% Annual Chance Flood Hazard"

zone_info = {
    "A":        ("#6A0DAD", "1% Annual Chance Flood Hazard (Zone A)"),
    "AE":       ("#6A0DAD", "1% Annual Chance Flood Hazard (Zone AE)"),
    "AO":       ("#6A0DAD", "1% Annual Chance Flood Hazard (Zone AO)"),
    "AH":       ("#6A0DAD", "1% Annual Chance Flood Hazard (Zone AH)"),
    "FLOODWAY": ("#c79fef", "Regulatory Floodway"),
    "VE":       ("#ffc0cb", "Coastal High Hazard Area"),
    "D":        ("#fdd9b5", "Area of Undetermined Flood Hazard"),
    "X500":     ("#DE8A29", "0.2% Annual Chance Flood Hazard"),  # This will use ZONE_SUBTY below
    "X":        (future_color, future_label)  # Updated: Zone X uses future conditions styling
}

levee_info = {
    "PROTECTED": ("#b0e0e6", "Area with Reduced Risk Due to Levee"),
    "RISK":      ("#f4a460", "Area with Risk Due to Levee"),
}

# -------------------------------------------------------------------
# 5. BUILD THE FOLIUM MAP
# -------------------------------------------------------------------
avg_lat = sum(lat for lat, lon in lat_lon_points) / len(lat_lon_points)
avg_lon = sum(lon for lat, lon in lat_lon_points) / len(lat_lon_points)
m = folium.Map(location=[avg_lat, avg_lon], zoom_start=7)

# Add markers for each point
for lat, lon in lat_lon_points:
    folium.Marker(
        location=[lat, lon],
        popup=f"Point: ({lat}, {lon})",
        icon=folium.Icon(color="red")
    ).add_to(m)

# -------------------------------------------------------------------
# 6. PROCESS EACH SHAPEFILE
# -------------------------------------------------------------------
for shp in shapefile_paths:
    base_name = os.path.splitext(os.path.basename(shp))[0].upper()
    
    try:
        gdf = gpd.read_file(shp).to_crs("EPSG:4326")

        # Convert timestamps to strings (fix serialization issue)
        for col in gdf.columns:
            if 'datetime' in str(gdf[col].dtype) or 'Timestamp' in str(gdf[col].dtype):
                gdf[col] = gdf[col].astype(str)

        # Ensure relevant columns exist
        if "FLD_ZONE" not in gdf.columns:
            gdf["FLD_ZONE"] = ""
        if "LEVEE_TYPE" not in gdf.columns:
            gdf["LEVEE_TYPE"] = ""
        if "ZONE_SUBTY" not in gdf.columns:
            gdf["ZONE_SUBTY"] = ""

        # ---------------- CASE 1: S_FLD_HAZ_AR (Standard Flood Hazard Zones) ----------------
        if "S_FLD_HAZ_AR" in base_name:
            for zone_code, (color, label) in zone_info.items():
                # For 0.2% hazard, use ZONE_SUBTY column with the associated value
                if zone_code == "X500":
                    subset = gdf[gdf["ZONE_SUBTY"] == '0.2 PCT ANNUAL CHANCE FLOOD HAZARD']
                else:
                    subset = gdf[gdf["FLD_ZONE"] == zone_code]
                if len(subset) == 0:
                    continue  # Skip empty layers
                
                folium.GeoJson(
                    data=subset.__geo_interface__,
                    name=label,  # Layer tagged with the flood hazard name
                    style_function=lambda feature, c=color: {
                        "fillColor": c,
                        "color": c,
                        "fillOpacity": 0.5,
                        "weight": 1
                    },
                    tooltip=GeoJsonTooltip(fields=["FLD_ZONE"], aliases=["Zone Code:"], sticky=False),
                    popup=GeoJsonPopup(fields=["FLD_ZONE"], aliases=["Zone Code:"])
                ).add_to(m)

        # ---------------- CASE 2: S_FLD_HAZ_FUTURE (Future 1% Chance) ----------------
        elif "S_FLD_HAZ_FUTURE" in base_name:
            if len(gdf) > 0:
                folium.GeoJson(
                    data=gdf.__geo_interface__,
                    name=future_label,
                    style_function=lambda feature: {
                        "fillColor": future_color,
                        "color": future_color,
                        "fillOpacity": 0.5,
                        "weight": 1
                    }
                ).add_to(m)

        # ---------------- CASE 3: S_LEVEE (Levee-Related Layers) ----------------
        elif "S_LEVEE" in base_name:
            for levee_code, (color, label) in levee_info.items():
                subset = gdf[gdf["LEVEE_TYPE"].str.upper() == levee_code]
                if len(subset) == 0:
                    continue
                
                folium.GeoJson(
                    data=subset.__geo_interface__,
                    name=label,  # Layer tagged correctly
                    style_function=lambda feature, c=color: {
                        "fillColor": c,
                        "color": c,
                        "fillOpacity": 0.5,
                        "weight": 1
                    }
                ).add_to(m)

        # ---------------- CASE 4: S_BFE (Base Flood Elevation Polyline Layers) ----------------
        elif "S_BFE" in base_name:
            # Style for BFE lines: thick red lines
            folium.GeoJson(
                data=gdf.__geo_interface__,
                name="Base Flood Elevation (BFE) Lines",
                style_function=lambda feature: {
                    "color": "#FF0000",
                    "weight": 3  # Adjust thickness as needed
                }
            ).add_to(m)

    except Exception as e:
        print(f"Error processing {shp}: {e}")

# -------------------------------------------------------------------
# 7. ADD LAYER CONTROL + CUSTOM LEGEND, THEN SAVE
# -------------------------------------------------------------------
folium.LayerControl().add_to(m)

# Custom legend (positioned in the lower left corner)
legend_html = '''
<div style="
position: fixed;
bottom: 50px;
left: 50px;
width: 280px;
height: 240px;
z-index:9999;
font-size:14px;
background-color: white;
opacity: 0.8;
padding: 10px;
border:2px solid grey;">
<strong>Map Legend</strong><br>
<hr style="margin:4px 0">
<i style="background:#6A0DAD; width: 18px; height: 18px; float: left; margin-right: 8px; opacity:0.7;"></i>
<span>1% Annual Chance Flood Hazard (Zones A, AE, AO, AH)</span><br>
<i style="background:#c79fef; width: 18px; height: 18px; float: left; margin-right: 8px; opacity:0.7;"></i>
<span>Regulatory Floodway</span><br>
<i style="background:#ffc0cb; width: 18px; height: 18px; float: left; margin-right: 8px; opacity:0.7;"></i>
<span>Special Floodway (Coastal High Hazard)</span><br>
<i style="background:#fdd9b5; width: 18px; height: 18px; float: left; margin-right: 8px; opacity:0.7;"></i>
<span>Area of Undetermined Flood Hazard</span><br>
<i style="background:#DE8A29; width: 18px; height: 18px; float: left; margin-right: 8px; opacity:0.7;"></i>
<span>0.2% Annual Chance Flood Hazard (Zone X500)</span><br>
<i style="background:{future_color}; width: 18px; height: 18px; float: left; margin-right: 8px; opacity:0.7;"></i>
<span>{future_label}</span><br>
<i style="background:#b0e0e6; width: 18px; height: 18px; float: left; margin-right: 8px; opacity:0.7;"></i>
<span>Area with Reduced Risk Due to Levee</span><br>
<i style="background:#f4a460; width: 18px; height: 18px; float: left; margin-right: 8px; opacity:0.7;"></i>
<span>Area with Risk Due to Levee</span><br>
<hr style="margin:4px 0">
<div style="display: flex; align-items: center;">
  <i style="border: 3px solid #FF0000; width: 18px; height: 18px; margin-right: 8px;"></i>
  <span>Base Flood Elevation (BFE) Lines</span>
</div>
</div>
'''.format(future_color=future_color, future_label=future_label)
m.get_root().html.add_child(folium.Element(legend_html))

output_html = "flood_hazard_map_Alameda_CA_06001C_BFE.html"
m.save(output_html)
print(f"Interactive map saved as '{output_html}'")

In [ ]:
import os
import zipfile
import glob
import geopandas as gpd
from shapely.geometry import Point
import folium
from folium.features import GeoJsonTooltip, GeoJsonPopup
import rasterio
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------------------------
# 1. EXTRACT FEMA ZIP (IF NEEDED)
# -------------------------------------------------------------------
zip_path = './FEMA_Downloads/06115C_20240605.zip'  # Update if needed
extract_path = './FEMA_Downloads/06001C_20241119'

if not os.path.exists(zip_path):
    raise FileNotFoundError(f"Zip file not found: {zip_path}")

if not os.path.exists(extract_path):
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(path=extract_path)
    print(f"Extracted files to {extract_path}")
else:
    print(f"Using existing extracted folder: {extract_path}")

# -------------------------------------------------------------------
# 2. CREATE A GEODATAFRAME FOR LAT/LON POINTS
# -------------------------------------------------------------------
lat_lon_points = [
    (34.052235, -118.243683),  # Los Angeles
    (37.774929, -122.419418),  # San Francisco
]

points_gdf = gpd.GeoDataFrame(
    geometry=[Point(lon, lat) for lat, lon in lat_lon_points],
    crs="EPSG:4326"
)

# -------------------------------------------------------------------
# 3. LOCATE SHAPEFILES
# -------------------------------------------------------------------
shapefile_paths = glob.glob(os.path.join(extract_path, "*.shp"))

if not shapefile_paths:
    print("No shapefiles found in the extraction folder.")
else:
    print(f"Found {len(shapefile_paths)} shapefile(s) in {extract_path}")

# -------------------------------------------------------------------
# 4. READ THE DEM DATA
# -------------------------------------------------------------------
dem_path = 'data/dem/alameda_clipped_dem.tif'  # Path to your DEM (assumed to be in meters)
dem = rasterio.open(dem_path)

# -------------------------------------------------------------------
# 5. FLOOD ZONE EXPLANATIONS & COLORS
# -------------------------------------------------------------------
future_color = "#ffa500"
future_label = "Future Conditions 1% Annual Chance Flood Hazard"

zone_info = {
    "A":        ("#6A0DAD", "1% Annual Chance Flood Hazard (Zone A)"),
    "AE":       ("#6A0DAD", "1% Annual Chance Flood Hazard (Zone AE)"),
    "AO":       ("#6A0DAD", "1% Annual Chance Flood Hazard (Zone AO)"),
    "AH":       ("#6A0DAD", "1% Annual Chance Flood Hazard (Zone AH)"),
    "FLOODWAY": ("#c79fef", "Regulatory Floodway"),
    "VE":       ("#ffc0cb", "Coastal High Hazard Area"),
    "D":        ("#fdd9b5", "Area of Undetermined Flood Hazard"),
    "X500":     ("#e8fea0", "0.2% Annual Chance Flood Hazard"),  # Uses ZONE_SUBTY below
    "X":        (future_color, future_label)  # Future conditions styling
}

levee_info = {
    "PROTECTED": ("#b0e0e6", "Area with Reduced Risk Due to Levee"),
    "RISK":      ("#f4a460", "Area with Risk Due to Levee"),
}

# -------------------------------------------------------------------
# 6. BUILD THE FOLIUM MAP
# -------------------------------------------------------------------
avg_lat = sum(lat for lat, lon in lat_lon_points) / len(lat_lon_points)
avg_lon = sum(lon for lat, lon in lat_lon_points) / len(lat_lon_points)
m = folium.Map(location=[avg_lat, avg_lon], zoom_start=7)

# Add markers for each point
for lat, lon in lat_lon_points:
    folium.Marker(
        location=[lat, lon],
        popup=f"Point: ({lat}, {lon})",
        icon=folium.Icon(color="red")
    ).add_to(m)

# -------------------------------------------------------------------
# 7. PROCESS EACH SHAPEFILE (INCLUDING DEM/FLOOD DEPTH CALCULATION)
# -------------------------------------------------------------------
# Conversion factor from feet to meters.
ft_to_m = 0.3048

for shp in shapefile_paths:
    base_name = os.path.splitext(os.path.basename(shp))[0].upper()
    
    try:
        gdf = gpd.read_file(shp).to_crs("EPSG:4326")

        # Convert timestamp columns to strings for serialization.
        for col in gdf.columns:
            if 'datetime' in str(gdf[col].dtype) or 'Timestamp' in str(gdf[col].dtype):
                gdf[col] = gdf[col].astype(str)

        # Ensure the expected columns exist.
        if "FLD_ZONE" not in gdf.columns:
            gdf["FLD_ZONE"] = ""
        if "LEVEE_TYPE" not in gdf.columns:
            gdf["LEVEE_TYPE"] = ""
        if "ZONE_SUBTY" not in gdf.columns:
            gdf["ZONE_SUBTY"] = ""
        if "ELEV" not in gdf.columns:
            gdf["ELEV"] = 0  # For BFE shapefile, this column should exist.
        
        # ---------------- CASE 1: S_FLD_HAZ_AR (Standard Flood Hazard Zones) ----------------
        if "S_FLD_HAZ_AR" in base_name:
            for zone_code, (color, label) in zone_info.items():
                # For 0.2% hazard, use ZONE_SUBTY column with the associated value.
                if zone_code == "X500":
                    subset = gdf[gdf["ZONE_SUBTY"] == '0.2 PCT ANNUAL CHANCE FLOOD HAZARD']
                else:
                    subset = gdf[gdf["FLD_ZONE"] == zone_code]
                if len(subset) == 0:
                    continue  # Skip if no matching features.
                
                folium.GeoJson(
                    data=subset.__geo_interface__,
                    name=label,
                    style_function=lambda feature, c=color: {
                        "fillColor": c,
                        "color": c,
                        "fillOpacity": 0.5,
                        "weight": 1
                    },
                    tooltip=GeoJsonTooltip(fields=["FLD_ZONE"], aliases=["Zone Code:"], sticky=False),
                    popup=GeoJsonPopup(fields=["FLD_ZONE"], aliases=["Zone Code:"])
                ).add_to(m)

        # ---------------- CASE 2: S_FLD_HAZ_FUTURE (Future 1% Chance) ----------------
        elif "S_FLD_HAZ_FUTURE" in base_name:
            if len(gdf) > 0:
                folium.GeoJson(
                    data=gdf.__geo_interface__,
                    name=future_label,
                    style_function=lambda feature: {
                        "fillColor": future_color,
                        "color": future_color,
                        "fillOpacity": 0.5,
                        "weight": 1
                    }
                ).add_to(m)

        # ---------------- CASE 3: S_LEVEE (Levee-Related Layers) ----------------
        elif "S_LEVEE" in base_name:
            for levee_code, (color, label) in levee_info.items():
                subset = gdf[gdf["LEVEE_TYPE"].str.upper() == levee_code]
                if len(subset) == 0:
                    continue
                
                folium.GeoJson(
                    data=subset.__geo_interface__,
                    name=label,
                    style_function=lambda feature, c=color: {
                        "fillColor": c,
                        "color": c,
                        "fillOpacity": 0.5,
                        "weight": 1
                    }
                ).add_to(m)

        # ---------------- CASE 4: S_BFE (Base Flood Elevation Polyline Layers) ----------------
        elif "S_BFE" in base_name:
            # Add the BFE lines to the folium map with thick red styling.
            folium.GeoJson(
                data=gdf.__geo_interface__,
                name="Base Flood Elevation (BFE) Lines",
                style_function=lambda feature: {
                    "color": "#FF0000",
                    "weight": 3
                }
            ).add_to(m)
            
            # Now perform DEM sampling and flood depth calculation for each cross section.
            for idx, row in gdf.iterrows():
                # Convert the BFE elevation from feet to meters.
                bfe_elev_m = row['ELEV'] * ft_to_m
                geom = row.geometry

                # Densify the cross section: sample 100 equally spaced points.
                num_points = 100
                distances = np.linspace(0, geom.length, num_points)
                sample_points = [geom.interpolate(distance) for distance in distances]
                coords = [(pt.x, pt.y) for pt in sample_points]

                # Sample the DEM at these coordinates.
                dem_samples = list(dem.sample(coords))
                dem_values = np.array([sample[0] for sample in dem_samples])

                # Calculate flood depth (DEM elevation minus BFE elevation).
                flood_depth = dem_values - bfe_elev_m
                flood_depth[flood_depth < 0] = 0  # Set negative values to zero.

                # Plot the flood depth along the cross section.
                plt.figure(figsize=(10, 5))
                plt.plot(np.linspace(0, geom.length, num_points), flood_depth, label='Flood Depth (m)')
                plt.xlabel('Distance along cross section (CRS units)')
                plt.ylabel('Flood Depth (m)')
                plt.title(f'Flood Depth Along BFE Cross Section (ID: {idx})')
                plt.legend()
                plt.grid(True)
                plt.show()

    except Exception as e:
        print(f"Error processing {shp}: {e}")

# -------------------------------------------------------------------
# 8. ADD LAYER CONTROL + CUSTOM LEGEND, THEN SAVE THE MAP
# -------------------------------------------------------------------
folium.LayerControl().add_to(m)

legend_html = f'''
<div style="
position: fixed;
bottom: 50px;
left: 50px;
width: 280px;
height: 240px;
z-index:9999;
font-size:14px;
background-color: white;
opacity: 0.8;
padding: 10px;
border:2px solid grey;">
<strong>Map Legend</strong><br>
<hr style="margin:4px 0">
<i style="background:#6A0DAD; width: 18px; height: 18px; float: left; margin-right: 8px; opacity:0.7;"></i>
<span>1% Annual Chance Flood Hazard (Zones A, AE, AO, AH)</span><br>
<i style="background:#c79fef; width: 18px; height: 18px; float: left; margin-right: 8px; opacity:0.7;"></i>
<span>Regulatory Floodway</span><br>
<i style="background:#ffc0cb; width: 18px; height: 18px; float: left; margin-right: 8px; opacity:0.7;"></i>
<span>Coastal High Hazard Area</span><br>
<i style="background:#fdd9b5; width: 18px; height: 18px; float: left; margin-right: 8px; opacity:0.7;"></i>
<span>Area of Undetermined Flood Hazard</span><br>
<i style="background:#e8fea0; width: 18px; height: 18px; float: left; margin-right: 8px; opacity:0.7;"></i>
<span>0.2% Annual Chance Flood Hazard (Zone X500)</span><br>
<i style="background:{future_color}; width: 18px; height: 18px; float: left; margin-right: 8px; opacity:0.7;"></i>
<span>{future_label}</span><br>
<i style="background:#b0e0e6; width: 18px; height: 18px; float: left; margin-right: 8px; opacity:0.7;"></i>
<span>Area with Reduced Risk Due to Levee</span><br>
<i style="background:#f4a460; width: 18px; height: 18px; float: left; margin-right: 8px; opacity:0.7;"></i>
<span>Area with Risk Due to Levee</span><br>
<hr style="margin:4px 0">
<div style="display: flex; align-items: center;">
  <i style="border: 3px solid #FF0000; width: 18px; height: 18px; margin-right: 8px;"></i>
  <span>Base Flood Elevation (BFE) Lines</span>
</div>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

output_html = "flood_hazard_map_Alameda_CA_06001C_BFE_with_DEM.html"
m.save(output_html)
print(f"Interactive map saved as '{output_html}'")


In [ ]:
import os
import zipfile
import glob
import geopandas as gpd
from shapely.geometry import Point
import folium
from folium.features import GeoJsonTooltip, GeoJsonPopup
import rasterio
import numpy as np
import matplotlib.pyplot as plt

# Helper function: Bresenham's Line Algorithm to get indices between two points.
def bresenham_line(x0, y0, x1, y1):
    """
    Generate grid cell indices between (x0,y0) and (x1,y1) using Bresenham's algorithm.
    Returns a list of (row, col) tuples.
    """
    x0 = int(round(x0))
    y0 = int(round(y0))
    x1 = int(round(x1))
    y1 = int(round(y1))
    
    dx = abs(x1 - x0)
    dy = abs(y1 - y0)
    x, y = x0, y0
    sx = -1 if x0 > x1 else 1
    sy = -1 if y0 > y1 else 1
    line_pixels = []
    if dx > dy:
        err = dx / 2.0
        while x != x1:
            line_pixels.append((y, x))
            err -= dy
            if err < 0:
                y += sy
                err += dx
            x += sx
        line_pixels.append((y, x))
    else:
        err = dy / 2.0
        while y != y1:
            line_pixels.append((y, x))
            err -= dx
            if err < 0:
                x += sx
                err += dy
            y += sy
        line_pixels.append((y, x))
    return line_pixels

# -------------------------------------------------------------------
# 1. EXTRACT FEMA ZIP (IF NEEDED)
# -------------------------------------------------------------------
zip_path = './FEMA_Downloads/06115C_20240605.zip'  # Update if needed
extract_path = './FEMA_Downloads/06001C_20241119'

if not os.path.exists(zip_path):
    raise FileNotFoundError(f"Zip file not found: {zip_path}")

if not os.path.exists(extract_path):
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(path=extract_path)
    print(f"Extracted files to {extract_path}")
else:
    print(f"Using existing extracted folder: {extract_path}")

# -------------------------------------------------------------------
# 2. CREATE A GEODATAFRAME FOR LAT/LON POINTS
# -------------------------------------------------------------------
lat_lon_points = [
    (34.052235, -118.243683),  # Los Angeles
    (37.774929, -122.419418),  # San Francisco
]

points_gdf = gpd.GeoDataFrame(
    geometry=[Point(lon, lat) for lat, lon in lat_lon_points],
    crs="EPSG:4326"
)

# -------------------------------------------------------------------
# 3. LOCATE SHAPEFILES
# -------------------------------------------------------------------
shapefile_paths = glob.glob(os.path.join(extract_path, "*.shp"))

if not shapefile_paths:
    print("No shapefiles found in the extraction folder.")
else:
    print(f"Found {len(shapefile_paths)} shapefile(s) in {extract_path}")

# -------------------------------------------------------------------
# 4. READ THE DEM DATA
# -------------------------------------------------------------------
dem_path = 'data/dem/alameda_clipped_dem.tif'  # Path to your DEM (assumed to be in meters)
dem = rasterio.open(dem_path)

# Create an empty flood depth array with exactly the same shape as the DEM.
flood_depth_raster = np.full((dem.height, dem.width), -9999, dtype=np.float32)
print(f"DEM dimensions: {dem.width} cols x {dem.height} rows")

# -------------------------------------------------------------------
# 5. FLOOD ZONE EXPLANATIONS & COLORS
# -------------------------------------------------------------------
future_color = "#ffa500"
future_label = "Future Conditions 1% Annual Chance Flood Hazard"

zone_info = {
    "A":        ("#6A0DAD", "1% Annual Chance Flood Hazard (Zone A)"),
    "AE":       ("#6A0DAD", "1% Annual Chance Flood Hazard (Zone AE)"),
    "AO":       ("#6A0DAD", "1% Annual Chance Flood Hazard (Zone AO)"),
    "AH":       ("#6A0DAD", "1% Annual Chance Flood Hazard (Zone AH)"),
    "FLOODWAY": ("#c79fef", "Regulatory Floodway"),
    "VE":       ("#ffc0cb", "Coastal High Hazard Area"),
    "D":        ("#fdd9b5", "Area of Undetermined Flood Hazard"),
    "X500":     ("#e8fea0", "0.2% Annual Chance Flood Hazard"),  # Uses ZONE_SUBTY below
    "X":        (future_color, future_label)  # Future conditions styling
}

levee_info = {
    "PROTECTED": ("#b0e0e6", "Area with Reduced Risk Due to Levee"),
    "RISK":      ("#f4a460", "Area with Risk Due to Levee"),
}

# -------------------------------------------------------------------
# 6. BUILD THE FOLIUM MAP
# -------------------------------------------------------------------
avg_lat = sum(lat for lat, lon in lat_lon_points) / len(lat_lon_points)
avg_lon = sum(lon for lat, lon in lat_lon_points) / len(lat_lon_points)
m = folium.Map(location=[avg_lat, avg_lon], zoom_start=7)

# Add markers for each point.
for lat, lon in lat_lon_points:
    folium.Marker(
        location=[lat, lon],
        popup=f"Point: ({lat}, {lon})",
        icon=folium.Icon(color="red")
    ).add_to(m)

# -------------------------------------------------------------------
# 7. PROCESS EACH SHAPEFILE (INCLUDING DEM/FLOOD DEPTH CALCULATION)
# -------------------------------------------------------------------
# Conversion factor from feet to meters.
ft_to_m = 0.3048

for shp in shapefile_paths:
    base_name = os.path.splitext(os.path.basename(shp))[0].upper()
    
    try:
        gdf = gpd.read_file(shp).to_crs("EPSG:4326")

        # Convert timestamp columns to strings for serialization.
        for col in gdf.columns:
            if 'datetime' in str(gdf[col].dtype) or 'Timestamp' in str(gdf[col].dtype):
                gdf[col] = gdf[col].astype(str)

        # Ensure the expected columns exist.
        if "FLD_ZONE" not in gdf.columns:
            gdf["FLD_ZONE"] = ""
        if "LEVEE_TYPE" not in gdf.columns:
            gdf["LEVEE_TYPE"] = ""
        if "ZONE_SUBTY" not in gdf.columns:
            gdf["ZONE_SUBTY"] = ""
        if "ELEV" not in gdf.columns:
            gdf["ELEV"] = 0  # For BFE shapefile, this column should exist.
        
        # ---------------- CASE 1: S_FLD_HAZ_AR (Standard Flood Hazard Zones) ----------------
        if "S_FLD_HAZ_AR" in base_name:
            for zone_code, (color, label) in zone_info.items():
                if zone_code == "X500":
                    subset = gdf[gdf["ZONE_SUBTY"] == '0.2 PCT ANNUAL CHANCE FLOOD HAZARD']
                else:
                    subset = gdf[gdf["FLD_ZONE"] == zone_code]
                if len(subset) == 0:
                    continue
                folium.GeoJson(
                    data=subset.__geo_interface__,
                    name=label,
                    style_function=lambda feature, c=color: {
                        "fillColor": c,
                        "color": c,
                        "fillOpacity": 0.5,
                        "weight": 1
                    },
                    tooltip=GeoJsonTooltip(fields=["FLD_ZONE"], aliases=["Zone Code:"], sticky=False),
                    popup=GeoJsonPopup(fields=["FLD_ZONE"], aliases=["Zone Code:"])
                ).add_to(m)

        # ---------------- CASE 2: S_FLD_HAZ_FUTURE (Future 1% Chance) ----------------
        elif "S_FLD_HAZ_FUTURE" in base_name:
            if len(gdf) > 0:
                folium.GeoJson(
                    data=gdf.__geo_interface__,
                    name=future_label,
                    style_function=lambda feature: {
                        "fillColor": future_color,
                        "color": future_color,
                        "fillOpacity": 0.5,
                        "weight": 1
                    }
                ).add_to(m)

        # ---------------- CASE 3: S_LEVEE (Levee-Related Layers) ----------------
        elif "S_LEVEE" in base_name:
            for levee_code, (color, label) in levee_info.items():
                subset = gdf[gdf["LEVEE_TYPE"].str.upper() == levee_code]
                if len(subset) == 0:
                    continue
                folium.GeoJson(
                    data=subset.__geo_interface__,
                    name=label,
                    style_function=lambda feature, c=color: {
                        "fillColor": c,
                        "color": c,
                        "fillOpacity": 0.5,
                        "weight": 1
                    }
                ).add_to(m)

        # ---------------- CASE 4: S_BFE (Base Flood Elevation Polyline Layers) ----------------
        elif "S_BFE" in base_name:
            folium.GeoJson(
                data=gdf.__geo_interface__,
                name="Base Flood Elevation (BFE) Lines",
                style_function=lambda feature: {
                    "color": "#FF0000",
                    "weight": 3
                }
            ).add_to(m)
            
            # Process each BFE feature for flood depth.
            for idx, row in gdf.iterrows():
                bfe_elev_m = row['ELEV'] * ft_to_m
                geom = row.geometry

                # Sample 100 equally spaced points along the cross section.
                num_points = 100
                distances = np.linspace(0, geom.length, num_points)
                sample_points = [geom.interpolate(distance) for distance in distances]
                coords = [(pt.x, pt.y) for pt in sample_points]

                # Sample DEM elevations at these coordinates.
                dem_samples = list(dem.sample(coords))
                dem_values = np.array([sample[0] for sample in dem_samples])

                # Compute flood depth (DEM elevation minus BFE elevation).
                flood_depth = dem_values - bfe_elev_m
                flood_depth[flood_depth < 0] = 0

                # Plot the flood depth along the cross section.
                plt.figure(figsize=(10, 5))
                plt.plot(np.linspace(0, geom.length, num_points), flood_depth, label='Flood Depth (m)')
                plt.xlabel('Distance along cross section (CRS units)')
                plt.ylabel('Flood Depth (m)')
                plt.title(f'Flood Depth Along BFE Cross Section (ID: {idx})')
                plt.legend()
                plt.grid(True)
                plt.show()

                # Get DEM grid indices for each sample point.
                grid_indices = []
                for pt in sample_points:
                    try:
                        row_idx, col_idx = dem.index(pt.x, pt.y)
                        grid_indices.append((row_idx, col_idx))
                    except Exception as e:
                        print(f"Skipping point at ({pt.x}, {pt.y}): {e}")
                        grid_indices.append(None)
                
                # Interpolate flood depth along the line segments between consecutive sample points.
                for i in range(len(sample_points) - 1):
                    start_idx = grid_indices[i]
                    end_idx = grid_indices[i+1]
                    if start_idx is None or end_idx is None:
                        continue
                    line_pixels = bresenham_line(start_idx[1], start_idx[0], end_idx[1], end_idx[0])
                    start_depth = flood_depth[i]
                    end_depth = flood_depth[i+1]
                    num_pixels = len(line_pixels)
                    if num_pixels < 2:
                        continue
                    for j, (r, c) in enumerate(line_pixels):
                        interp_depth = start_depth + (end_depth - start_depth) * (j / (num_pixels - 1))
                        # Only assign if the pixel is within the DEM's boundaries.
                        if 0 <= r < flood_depth_raster.shape[0] and 0 <= c < flood_depth_raster.shape[1]:
                            flood_depth_raster[r, c] = interp_depth
                        else:
                            print(f"Skipping pixel at (row {r}, col {c}) as it is out of bounds.")

    except Exception as e:
        print(f"Error processing {shp}: {e}")

# -------------------------------------------------------------------
# Write the flood depth raster to a GeoTIFF file using the DEM's profile.
# -------------------------------------------------------------------
out_path = "flood_depth.tif"
out_profile = dem.profile.copy()
out_profile.update(dtype=rasterio.float32, nodata=-9999, count=1)
with rasterio.open(out_path, 'w', **out_profile) as dst:
    dst.write(flood_depth_raster, 1)
print(f"Flood depth raster saved as '{out_path}'")
print(f"Output raster dimensions: {out_profile['width']} cols x {out_profile['height']} rows")

# -------------------------------------------------------------------
# 8. ADD LAYER CONTROL + CUSTOM LEGEND, THEN SAVE THE MAP
# -------------------------------------------------------------------
folium.LayerControl().add_to(m)

legend_html = f'''
<div style="
position: fixed;
bottom: 50px;
left: 50px;
width: 280px;
height: 240px;
z-index:9999;
font-size:14px;
background-color: white;
opacity:0.8;
padding: 10px;
border:2px solid grey;">
<strong>Map Legend</strong><br>
<hr style="margin:4px 0">
<i style="background:#6A0DAD; width:18px; height:18px; float:left; margin-right:8px; opacity:0.7;"></i>
<span>1% Annual Chance Flood Hazard (Zones A, AE, AO, AH)</span><br>
<i style="background:#c79fef; width:18px; height:18px; float:left; margin-right:8px; opacity:0.7;"></i>
<span>Regulatory Floodway</span><br>
<i style="background:#ffc0cb; width:18px; height:18px; float:left; margin-right:8px; opacity:0.7;"></i>
<span>Coastal High Hazard Area</span><br>
<i style="background:#fdd9b5; width:18px; height:18px; float:left; margin-right:8px; opacity:0.7;"></i>
<span>Area of Undetermined Flood Hazard</span><br>
<i style="background:#e8fea0; width:18px; height:18px; float:left; margin-right:8px; opacity:0.7;"></i>
<span>0.2% Annual Chance Flood Hazard (Zone X500)</span><br>
<i style="background:{future_color}; width:18px; height:18px; float:left; margin-right:8px; opacity:0.7;"></i>
<span>{future_label}</span><br>
<i style="background:#b0e0e6; width:18px; height:18px; float:left; margin-right:8px; opacity:0.7;"></i>
<span>Area with Reduced Risk Due to Levee</span><br>
<i style="background:#f4a460; width:18px; height:18px; float:left; margin-right:8px; opacity:0.7;"></i>
<span>Area with Risk Due to Levee</span><br>
<hr style="margin:4px 0">
<div style="display:flex; align-items:center;">
  <i style="border:3px solid #FF0000; width:18px; height:18px; margin-right:8px;"></i>
  <span>Base Flood Elevation (BFE) Lines</span>
</div>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

output_html = "flood_hazard_map_Alameda_CA_06001C_BFE_with_DEM.html"
m.save(output_html)
print(f"Interactive map saved as '{output_html}'")


In [ ]:
import os
import zipfile
import glob
import geopandas as gpd
from shapely.geometry import Point
import folium
from folium.features import GeoJsonTooltip, GeoJsonPopup
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

# -------------------------------------------------------------------
# 1. EXTRACT FEMA ZIP (IF NEEDED)
# -------------------------------------------------------------------
zip_path = './FEMA_Downloads/06115C_20240605.zip'  # Update if needed
extract_path = './FEMA_Downloads/06001C_20241119'

if not os.path.exists(zip_path):
    raise FileNotFoundError(f"Zip file not found: {zip_path}")

if not os.path.exists(extract_path):
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(path=extract_path)
    print(f"Extracted files to {extract_path}")
else:
    print(f"Using existing extracted folder: {extract_path}")

# -------------------------------------------------------------------
# 2. CREATE A GEODATAFRAME FOR LAT/LON POINTS
# -------------------------------------------------------------------
lat_lon_points = [
    (34.052235, -118.243683),  # Los Angeles
    (37.774929, -122.419418),  # San Francisco
]

points_gdf = gpd.GeoDataFrame(
    geometry=[Point(lon, lat) for lat, lon in lat_lon_points],
    crs="EPSG:4326"
)

# -------------------------------------------------------------------
# 3. LOCATE SHAPEFILES
# -------------------------------------------------------------------
shapefile_paths = glob.glob(os.path.join(extract_path, "*.shp"))
if not shapefile_paths:
    print("No shapefiles found in the extraction folder.")
else:
    print(f"Found {len(shapefile_paths)} shapefile(s) in {extract_path}")

# -------------------------------------------------------------------
# 4. READ THE DEM DATA
# -------------------------------------------------------------------
dem_path = 'data/dem/alameda_clipped_dem.tif'  # Update with your DEM path
dem = rasterio.open(dem_path)

# Create an empty flood depth array with the same shape as the DEM.
flood_depth_raster = np.full((dem.height, dem.width), -9999, dtype=np.float32)
print(f"DEM dimensions: {dem.width} cols x {dem.height} rows")

# -------------------------------------------------------------------
# 5. FLOOD ZONE EXPLANATIONS & COLORS
# -------------------------------------------------------------------
future_color = "#ffa500"
future_label = "Future Conditions 1% Annual Chance Flood Hazard"

zone_info = {
    "A":        ("#6A0DAD", "1% Annual Chance Flood Hazard (Zone A)"),
    "AE":       ("#6A0DAD", "1% Annual Chance Flood Hazard (Zone AE)"),
    "AO":       ("#6A0DAD", "1% Annual Chance Flood Hazard (Zone AO)"),
    "AH":       ("#6A0DAD", "1% Annual Chance Flood Hazard (Zone AH)"),
    "FLOODWAY": ("#c79fef", "Regulatory Floodway"),
    "VE":       ("#ffc0cb", "Coastal High Hazard Area"),
    "D":        ("#fdd9b5", "Area of Undetermined Flood Hazard"),
    "X500":     ("#e8fea0", "0.2% Annual Chance Flood Hazard"),
    "X":        (future_color, future_label)
}

levee_info = {
    "PROTECTED": ("#b0e0e6", "Area with Reduced Risk Due to Levee"),
    "RISK":      ("#f4a460", "Area with Risk Due to Levee"),
}

# -------------------------------------------------------------------
# 6. BUILD THE FOLIUM MAP
# -------------------------------------------------------------------
avg_lat = sum(lat for lat, lon in lat_lon_points) / len(lat_lon_points)
avg_lon = sum(lon for lat, lon in lat_lon_points) / len(lat_lon_points)
m = folium.Map(location=[avg_lat, avg_lon], zoom_start=7)

for lat, lon in lat_lon_points:
    folium.Marker(
        location=[lat, lon],
        popup=f"Point: ({lat}, {lon})",
        icon=folium.Icon(color="red")
    ).add_to(m)

# -------------------------------------------------------------------
# 7. PROCESS EACH SHAPEFILE (INCLUDING FLOOD DEPTH CALCULATION)
# -------------------------------------------------------------------
ft_to_m = 0.3048

# Lists to store sample point coordinates and flood depth values.
all_sample_coords = []
all_sample_values = []

for shp in shapefile_paths:
    base_name = os.path.splitext(os.path.basename(shp))[0].upper()
    try:
        gdf = gpd.read_file(shp).to_crs("EPSG:4326")
        for col in gdf.columns:
            if 'datetime' in str(gdf[col].dtype) or 'Timestamp' in str(gdf[col].dtype):
                gdf[col] = gdf[col].astype(str)
        if "FLD_ZONE" not in gdf.columns:
            gdf["FLD_ZONE"] = ""
        if "LEVEE_TYPE" not in gdf.columns:
            gdf["LEVEE_TYPE"] = ""
        if "ZONE_SUBTY" not in gdf.columns:
            gdf["ZONE_SUBTY"] = ""
        if "ELEV" not in gdf.columns:
            gdf["ELEV"] = 0

        if "S_BFE" in base_name:
            folium.GeoJson(
                data=gdf.__geo_interface__,
                name="Base Flood Elevation (BFE) Lines",
                style_function=lambda feature: {"color": "#FF0000", "weight": 3}
            ).add_to(m)
            
            for idx, row in gdf.iterrows():
                bfe_elev_m = row['ELEV'] * ft_to_m
                geom = row.geometry
                num_points = 100
                distances = np.linspace(0, geom.length, num_points)
                sample_points = [geom.interpolate(distance) for distance in distances]
                coords = [(pt.x, pt.y) for pt in sample_points]
                dem_samples = list(dem.sample(coords))
                dem_values = np.array([sample[0] for sample in dem_samples])
                flood_depth = dem_values - bfe_elev_m
                flood_depth[flood_depth < 0] = 0

                plt.figure(figsize=(10, 5))
                plt.plot(np.linspace(0, geom.length, num_points), flood_depth, label='Flood Depth (m)')
                plt.xlabel('Distance (CRS units)')
                plt.ylabel('Flood Depth (m)')
                plt.title(f'Flood Depth Along BFE Cross Section (ID: {idx})')
                plt.legend()
                plt.grid(True)
                plt.show()

                for pt, depth in zip(sample_points, flood_depth):
                    try:
                        row_idx, col_idx = dem.index(pt.x, pt.y)
                        # Convert grid indices back to real-world coordinates.
                        x, y = rasterio.transform.xy(dem.transform, row_idx, col_idx)
                        all_sample_coords.append((x, y))
                        all_sample_values.append(depth)
                    except Exception as e:
                        print(f"Skipping point at ({pt.x}, {pt.y}): {e}")
        else:
            # Process other shapefile cases if needed.
            pass

    except Exception as e:
        print(f"Error processing {shp}: {e}")

# -------------------------------------------------------------------
# INTERPOLATE USING NEAREST-NEIGHBOR TO FILL THE ENTIRE GRID
# -------------------------------------------------------------------
cols, rows = np.meshgrid(np.arange(dem.width), np.arange(dem.height))
grid_x, grid_y = rasterio.transform.xy(dem.transform, rows, cols, offset='center')
grid_x = np.array(grid_x)
grid_y = np.array(grid_y)

if all_sample_coords and all_sample_values:
    sample_coords = np.array(all_sample_coords)
    sample_values = np.array(all_sample_values)
    
    # Use nearest neighbor so that every grid cell is assigned a value.
    interp_grid = griddata(
        sample_coords,
        sample_values,
        (grid_x, grid_y),
        method='nearest'
    )
    # Reshape the output to the DEM dimensions.
    flood_depth_raster = interp_grid.reshape((dem.height, dem.width)).astype(np.float32)
else:
    print("No sample points collected. Flood depth raster remains unchanged.")

# -------------------------------------------------------------------
# WRITE THE RASTER WITH THE SAME PROFILE AS THE DEM
# -------------------------------------------------------------------
out_path = "flood_depth.tif"
out_profile = dem.profile.copy()
out_profile.update(dtype=rasterio.float32, nodata=-9999, count=1)
with rasterio.open(out_path, 'w', **out_profile) as dst:
    dst.write(flood_depth_raster, 1)
print(f"Flood depth raster saved as '{out_path}'")
print(f"Output raster dimensions: {out_profile['width']} cols x {out_profile['height']} rows")

# -------------------------------------------------------------------
# ADD LAYER CONTROL + LEGEND, THEN SAVE THE MAP
# -------------------------------------------------------------------
folium.LayerControl().add_to(m)

legend_html = f'''
<div style="
position: fixed;
bottom: 50px;
left: 50px;
width: 280px;
height: 240px;
z-index:9999;
font-size:14px;
background-color: white;
opacity:0.8;
padding: 10px;
border:2px solid grey;">
<strong>Map Legend</strong><br>
<hr style="margin:4px 0">
<i style="background:#6A0DAD; width:18px; height:18px; float:left; margin-right:8px; opacity:0.7;"></i>
<span>1% Annual Chance Flood Hazard (Zones A, AE, AO, AH)</span><br>
<i style="background:#c79fef; width:18px; height:18px; float:left; margin-right:8px; opacity:0.7;"></i>
<span>Regulatory Floodway</span><br>
<i style="background:#ffc0cb; width:18px; height:18px; float:left; margin-right:8px; opacity:0.7;"></i>
<span>Coastal High Hazard Area</span><br>
<i style="background:#fdd9b5; width:18px; height:18px; float:left; margin-right:8px; opacity:0.7;"></i>
<span>Area of Undetermined Flood Hazard</span><br>
<i style="background:#e8fea0; width:18px; height:18px; float:left; margin-right:8px; opacity:0.7;"></i>
<span>0.2% Annual Chance Flood Hazard (Zone X500)</span><br>
<i style="background:{future_color}; width:18px; height:18px; float:left; margin-right:8px; opacity:0.7;"></i>
<span>{future_label}</span><br>
<i style="background:#b0e0e6; width:18px; height:18px; float:left; margin-right:8px; opacity:0.7;"></i>
<span>Area with Reduced Risk Due to Levee</span><br>
<i style="background:#f4a460; width:18px; height:18px; float:left; margin-right:8px; opacity:0.7;"></i>
<span>Area with Risk Due to Levee</span><br>
<hr style="margin:4px 0">
<div style="display:flex; align-items:center;">
  <i style="border:3px solid #FF0000; width:18px; height:18px; margin-right:8px;"></i>
  <span>Base Flood Elevation (BFE) Lines</span>
</div>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

output_html = "flood_hazard_map_Alameda_CA_06001C_BFE_with_DEM.html"
m.save(output_html)
print(f"Interactive map saved as '{output_html}'")
